# CIS 531/731: Programming Techniques for Data Science and Analytics
## Lab 10: Model Metrics, Probability Calibration & Error Analysis

--- 

### **Module 10 / Lecture 33 Hands-on Lab & Homework Alignment**
**Course:** CIS 531/731 Programming Techniques for Data Science and Analytics  
**Graded Assignment Alignment:** **MP 10: Data Science Interpretability & Metrics Scoring**  
**Topic:** Implementing Model Metrics, Probability Calibration (Platt Scaling & Isotonic Regression), and Systematic Error Dissection in Python  
**Associated Reading:** `scikit-learn` *Metrics and Scoring*; `scikit-learn` *Probability Calibration Guide*; Sentence Transformers *Semantic Search*

--- 

## **Lab Overview & Objectives**

In this lab, you will translate the model evaluation and calibration concepts from **Lectures 31–33** into reproducible Python data science pipelines. This lab provides foundational code structures directly applicable to your **MP 10** submission.

By the end of this lab, you will be able to:
1. **Evaluate Imbalanced Classifiers:** Compute Precision, Recall, F1-score, Brier Score Loss, and ROC-AUC under class-imbalanced conditions.
2. **Implement Probability Calibration:** Use `scikit-learn`'s `CalibratedClassifierCV` to apply **Platt Scaling (Sigmoid)** and **Isotonic Regression** to uncalibrated tree-based models.
3. **Plot Reliability Diagrams:** Generate calibration curves to visually diagnose overconfidence and underconfidence.
4. **Perform Systematic Error Dissection:** Programmatically slice validation data to isolate worst-case residuals and highest-loss predictions.

--- 
## **Task 1: Environment Setup & Dataset Generation**

Run the block below to generate a synthetic imbalanced classification dataset simulating real-world analytics data.

In [ ]:
# Task 1: Environment Setup & Imbalanced Synthetic Dataset Generation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve
)
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV

# Set seed for reproducibility
np.random.seed(531)

# Generate 10,000 samples with 90:10 class imbalance (Imbalanced Analytics Benchmark)
X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    weights=[0.90, 0.10],
    random_state=531
)

# Stratified Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=531)

print(f"[Dataset Ready] Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train Class Ratio (Positive Class): {np.mean(y_train):.3f}")
print(f"Test Class Ratio (Positive Class):  {np.mean(y_test):.3f}")

--- 
## **Task 2: Uncalibrated Baseline & Probability Diagnostic**

Train an uncalibrated `GradientBoostingClassifier` and compute raw metrics and initial Brier score loss.

In [ ]:
# Task 2: Train Uncalibrated Baseline Model
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=531)
gb_model.fit(X_train, y_train)

# Predict Raw Probabilities and Class Labels
y_prob_uncal = gb_model.predict_proba(X_test)[:, 1]
y_pred_uncal = gb_model.predict(X_test)

# Evaluate Baseline Performance Metrics
auc_uncal = roc_auc_score(y_test, y_prob_uncal)
brier_uncal = brier_score_loss(y_test, y_prob_uncal)

print("=== Uncalibrated Baseline Metrics ===")
print(f"ROC-AUC Score:      {auc_uncal:.4f}")
print(f"Brier Score Loss:   {brier_uncal:.4f} (Lower is better)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_uncal, digits=4))

--- 
## **Task 3: Implement Platt Scaling & Isotonic Regression**

Apply `CalibratedClassifierCV` using both **Platt Scaling (`method='sigmoid'`)** and **Isotonic Regression (`method='isotonic'`)** to calibrate the decision outputs.

In [ ]:
# Task 3: Probability Calibration with Platt Scaling and Isotonic Regression

# 1. Platt Scaling (Sigmoid Model)
calibrated_platt = CalibratedClassifierCV(estimator=GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=531), method='sigmoid', cv=5)
calibrated_platt.fit(X_train, y_train)
y_prob_platt = calibrated_platt.predict_proba(X_test)[:, 1]

# 2. Isotonic Regression
calibrated_iso = CalibratedClassifierCV(estimator=GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=531), method='isotonic', cv=5)
calibrated_iso.fit(X_train, y_train)
y_prob_iso = calibrated_iso.predict_proba(X_test)[:, 1]

# Brier Score Comparison
brier_platt = brier_score_loss(y_test, y_prob_platt)
brier_iso = brier_score_loss(y_test, y_prob_iso)

print("=== Brier Score Loss Comparison ===")
print(f"Uncalibrated Baseline:  {brier_uncal:.5f}")
print(f"Platt Scaling (Sigmoid): {brier_platt:.5f}")
print(f"Isotonic Regression:    {brier_iso:.5f}")

In [ ]:
# Task 3b: Plot Reliability Curves (Calibration Curves)
fig, ax = plt.subplots(figsize=(8, 6))

CalibrationDisplay.from_predictions(y_test, y_prob_uncal, n_bins=10, name="Uncalibrated GB", ax=ax)
CalibrationDisplay.from_predictions(y_test, y_prob_platt, n_bins=10, name="Platt Scaling", ax=ax)
CalibrationDisplay.from_predictions(y_test, y_prob_iso, n_bins=10, name="Isotonic Regression", ax=ax)

ax.set_title("Reliability Diagram (Calibration Curves) - CIS 531/731 Lab 10", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

--- 
## **Task 4: Systematic Error Dissection & Residual Slicing**

Extract worst-case prediction errors to inspect feature slices where the model exhibits highest log-loss.

In [ ]:
# Task 4: Error Slicing & Dissection
test_df = pd.DataFrame(X_test, columns=[f"feature_{i}" for i in range(20)])
test_df['y_true'] = y_test
test_df['y_prob'] = y_prob_iso

# Calculate Binary Cross-Entropy (Log-Loss) per sample
eps = 1e-15
probs_clipped = np.clip(y_prob_iso, eps, 1 - eps)
test_df['sample_loss'] = -(test_df['y_true'] * np.log(probs_clipped) + (1 - test_df['y_true']) * np.log(1 - probs_clipped))

# Extract Top 10 Worst False Positives & Top 10 Worst False Negatives
worst_false_positives = test_df[(test_df['y_true'] == 0)].sort_values(by='sample_loss', ascending=False).head(5)
worst_false_negatives = test_df[(test_df['y_true'] == 1)].sort_values(by='sample_loss', ascending=False).head(5)

print("=== Top 5 Worst False Positives (High Confidence False Alarms) ===")
print(worst_false_positives[['y_true', 'y_prob', 'sample_loss', 'feature_0', 'feature_1']])

print("\n=== Top 5 Worst False Negatives (High Confidence Misses) ===")
print(worst_false_negatives[['y_true', 'y_prob', 'sample_loss', 'feature_0', 'feature_1']])

--- 
## **Lab 10 Deliverables & MP 10 Submission Checklist**

For your **MP 10** submission, make sure to:

- [ ] **Complete Code Implementation:** Ensure `CalibratedClassifierCV` is correctly integrated into your machine learning pipelines.
- [ ] **Generate Reliability Plots:** Include saved plots comparing uncalibrated vs. calibrated reliability curves.
- [ ] **Report Metric Improvements:** Log baseline vs. calibrated Brier scores and ROC-AUC scores in your MP 10 report.
- [ ] **Submit Error Analysis Log:** Export the top 10 error-dissection table to your student submission repository.